# MuseumSCAT: fine-tune doctr's Danish-vocab recognizer

Before running: attach the private dataset `museumscat-danish-recognizer-crops` as input
(Notebook Settings → Input → search for it → Add), and under Settings enable
**Accelerator = GPU T4 x2 (or P100)** and **Internet = On**.

**One-time setup for auto-save (Cell 7):** the last cell pushes both checkpoints to the
permanent private Kaggle Dataset `museumscat-danish-recognizer-checkpoints` automatically
at the end of the run — no manual download/save needed, safe even if you don't come back
to the session. This needs Kaggle API credentials attached as notebook Secrets once:
1. https://www.kaggle.com/settings → API → Create New Token (downloads a `kaggle.json`
   with your username + key).
2. This notebook's top toolbar → **Add-ons → Secrets** → add `KAGGLE_USERNAME` and
   `KAGGLE_KEY` (values from that file) → make sure both show as Attached.

**Two-stage recipe, v3** (see `docs/superpowers/plans/2026-07-24-danish-recognizer-finetune.md`,
"Fine-tune result" section for the full history): v1 (20 epochs) left the head
undertrained (every date → MISSING). v2 (150/50 epoch ceilings, patience 15/10) fixed
that — locality NED beat baseline, date NED didn't — but finished in ~1 minute, meaning
early stopping likely tripped on validation-loss noise (only ~4-7 batches/epoch), not
real convergence.

This version pushes ceilings and patience much further still (1500/800 epochs, patience
120/100), same architecture (`crnn_vgg16_bn`) — duration/patience is the safer lever here
than model capacity, since a bigger architecture on only 391 training crops is more
likely to overfit harder, not generalize better. Batch size 16, AdamW + weight decay in
Stage 2, and `--amp` carried over unchanged from v2.

**Worst case (no early stop) this runs well past an hour** — if it's still going and you
want to cap GPU usage, stop the session manually rather than waiting for the ceiling
(Cell 7 only runs after Cell 5/6 finish, so an interrupted Stage 2 won't auto-save partial
results — that's expected).

In [ ]:
!pip install -q "python-doctr[torch]==1.0.1" matplotlib
!mkdir -p doctr_recognition
!curl -s -o doctr_recognition/train.py "https://raw.githubusercontent.com/mindee/doctr/v1.0.1/references/recognition/train.py"
!curl -s -o doctr_recognition/utils.py "https://raw.githubusercontent.com/mindee/doctr/v1.0.1/references/recognition/utils.py"

In [ ]:
import os

INPUT_DIR = "/kaggle/input/datasets/matiaspoullain/museumscat-danish-recognizer-crops"
TRAIN_PATH = os.path.join(INPUT_DIR, "train")
VAL_PATH = os.path.join(INPUT_DIR, "val")
assert os.path.isfile(os.path.join(TRAIN_PATH, "labels.json"))
assert os.path.isfile(os.path.join(VAL_PATH, "labels.json"))

In [ ]:
!cd doctr_recognition && python train.py crnn_vgg16_bn \
    --vocab danish --pretrained --freeze-backbone --amp \
    --train_path {TRAIN_PATH} --val_path {VAL_PATH} \
    --epochs 1500 --batch_size 16 --lr 0.001 \
    --early-stop --early-stop-epochs 120 --early-stop-delta 0.0005 \
    --output_dir /kaggle/working --name danish_crnn_vgg16_bn_stage1

In [ ]:
import os

assert os.path.isfile("/kaggle/working/danish_crnn_vgg16_bn_stage1.pt")
print("stage 1 checkpoint ready: /kaggle/working/danish_crnn_vgg16_bn_stage1.pt")

In [ ]:
!cd doctr_recognition && python train.py crnn_vgg16_bn \
    --vocab danish --resume /kaggle/working/danish_crnn_vgg16_bn_stage1.pt --amp \
    --train_path {TRAIN_PATH} --val_path {VAL_PATH} \
    --epochs 800 --batch_size 16 --lr 0.0001 --optim adamw --wd 0.0001 \
    --early-stop --early-stop-epochs 100 --early-stop-delta 0.0005 \
    --output_dir /kaggle/working --name danish_crnn_vgg16_bn_stage2

In [ ]:
import os

assert os.path.isfile("/kaggle/working/danish_crnn_vgg16_bn_stage2.pt")
print("final checkpoint ready to download: /kaggle/working/danish_crnn_vgg16_bn_stage2.pt")

In [ ]:
import datetime
import json
import os
import shutil

from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["KAGGLE_USERNAME"] = secrets.get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = secrets.get_secret("KAGGLE_KEY")

OUT_DIR = "/kaggle/working/checkpoints_dataset"
os.makedirs(OUT_DIR, exist_ok=True)
shutil.copy("/kaggle/working/danish_crnn_vgg16_bn_stage1.pt", OUT_DIR)
shutil.copy("/kaggle/working/danish_crnn_vgg16_bn_stage2.pt", OUT_DIR)
with open(os.path.join(OUT_DIR, "dataset-metadata.json"), "w") as f:
    json.dump(
        {
            "title": "MuseumSCAT Danish recognizer checkpoints",
            "id": "matiaspoullain/museumscat-danish-recognizer-checkpoints",
            "licenses": [{"name": "other"}],
        },
        f,
    )

version_message = f"auto-save from notebook run at {datetime.datetime.utcnow().isoformat()}"
!kaggle datasets version -p {OUT_DIR} -m "{version_message}" --dir-mode zip